# Adaptive Query Planning: Local Hugging Face Experiments
Use **Runtime > Change runtime type > T4 GPU** where available. This notebook downloads one model at a time, writes incremental records to Google Drive, and never requires an OpenAI key.

In [ ]:
!nvidia-smi || true
import torch, platform
print({'python': platform.python_version(), 'cuda_available': torch.cuda.is_available(), 'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/AQP_IRET_Research_Results')
for name in ['configs', 'logs', 'model_metadata', 'raw_results', 'traces', 'summaries', 'comparisons', 'plots', 'failures']:
    (DRIVE_ROOT / name).mkdir(parents=True, exist_ok=True)
print('Drive output root:', DRIVE_ROOT)

In [ ]:
REPOSITORY_URL = 'https://github.com/MSaiKiran9/aqp_iret.git'
!git clone $REPOSITORY_URL
%cd aqp_iret
!pip -q install -r requirements.txt

In [ ]:
import yaml
from pathlib import Path
MODELS = [
    'Qwen/Qwen2.5-1.5B-Instruct',
    'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    'HuggingFaceTB/SmolLM2-1.7B-Instruct',
    'microsoft/Phi-3.5-mini-instruct',
]
SMOKE_MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
DATASET_SIZE = 5       # use 1 for the fastest first check
RUN_FULL_EVALUATION = False
config = yaml.safe_load(Path('config.yaml').read_text())
config['dataset'] = {'name': 'hotpotqa', 'split': 'validation', 'sample_size': DATASET_SIZE}
config['llm']['enabled'] = True
config['llm']['provider'] = 'huggingface_local'
config['llm']['device'] = 'auto'
config['llm']['precision'] = 'auto'
config['llm']['models'] = [item for item in config['llm']['models'] if item['id'] in MODELS]
Path('colab_hf_config.yaml').write_text(yaml.safe_dump(config, sort_keys=False))
print('Selected models:', [item['id'] for item in config['llm']['models']])

In [ ]:
# Offline tests do not download models.
!python -m pytest tests -q

In [ ]:
# Actual local-HF smoke test: one question, one model, all selected methods.
!python experiments/run_hf_experiments.py --config colab_hf_config.yaml --output $DRIVE_ROOT --models $SMOKE_MODEL --smoke
print('Smoke output saved under:', DRIVE_ROOT)

In [ ]:
# This cell is intentionally opt-in. It processes models sequentially and resumes records already written for the same RUN_ID.
if RUN_FULL_EVALUATION:
    import subprocess, uuid
    RUN_ID = uuid.uuid4().hex[:12]
    for model_id in MODELS:
        subprocess.run(['python', 'experiments/run_hf_experiments.py', '--config', 'colab_hf_config.yaml', '--output', str(DRIVE_ROOT), '--models', model_id, '--run-id', RUN_ID], check=False)
        import gc, torch
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    print('Full run id:', RUN_ID)
else:
    print('Full evaluation not started. Set RUN_FULL_EVALUATION=True after reviewing the smoke result.')

In [ ]:
import glob, pandas as pd
comparison = DRIVE_ROOT / 'comparisons' / 'model_method_comparison.csv'
if comparison.exists(): display(pd.read_csv(comparison))
dashboard = DRIVE_ROOT / 'plots' / 'comparison_dashboard.png'
if dashboard.exists():
    from IPython.display import Image, display
    display(Image(str(dashboard)))
print('All outputs remain in:', DRIVE_ROOT)